# SureSuite Public API — notebook quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suresuite/suresuite-v001-3-f6699a5b/blob/main/public/notebooks/suresuite_api_quickstart.ipynb)

This notebook drives **SureSuite** end-to-end through the public `/v1` API — the same
authenticated, scope-checked, rate-limited gateway every external integration uses.
It covers **every v1 use case**:

| § | Use case | Scopes needed |
|---|---|---|
| 1 | Connect & list your projects | `read:data` |
| 2 | Input data — dataset versions & freezing a provenance snapshot | `read:data`, `write:data` |
| 3 | Explore the engine policy catalog | `read:policies` |
| 4 | Read & edit the project's policy configuration | `read:policies`, `write:policies` |
| 5 | Snapshot an immutable policy version | `write:policies` |
| 6 | Scenarios — list & create (baseline + disruption) | `read:runs`, `write:runs` |
| 7 | Dispatch a simulation run & poll to completion | `write:runs`, `read:runs` |
| 8 | Results — aggregate KPIs, per-replication analysis, plots | `read:runs` |
| 9 | Model-credibility (validation) status | `read:runs` |
| 10 | Experiment: compare policy A vs B | all of the above |
| 11 | Experiment: disruption resilience test | all of the above |
| 12 | Operations — cancel, extend, error handling | `write:runs` |

**Before you start** — on the app's **`/developer`** page:

1. Create an API key (a `sk_test_…` key is recommended while you experiment).
2. Open the **Notebook** tab, pick your project, and copy its config into the
   CONFIG cell below (or download this notebook from there — then it's pre-filled).


In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
# Auto-filled when this notebook is downloaded from the app's /developer →
# Notebook tab. Otherwise, copy the values from that tab.
BASE_URL = "https://wckdrutwkytwcomrlpib.supabase.co/functions/v1/api/v1"
PROJECT_ID = ""          # required — the project this notebook works in
SCENARIO_ID = ""         # optional — leave "" and §6 picks (or creates) one
POLICY_VERSION_ID = ""   # optional — leave "" and §5 snapshots a fresh one

## API key

The key is a secret — never paste it into a cell. Pick whichever applies:

- **Google Colab** — open the 🔑 **Secrets** panel (left sidebar), add a secret named
  `SURESUITE_API_KEY`, and grant this notebook access.
- **Local Jupyter** — export `SURESUITE_API_KEY` in your environment before starting.
- **Fallback** — the cell prompts you interactively (input is hidden, not stored).


In [ ]:
import os, getpass

API_KEY = None
try:
    from google.colab import userdata  # Colab: 🔑 Secrets panel
    API_KEY = userdata.get("SURESUITE_API_KEY")
except Exception:
    pass
API_KEY = API_KEY or os.environ.get("SURESUITE_API_KEY") or getpass.getpass("SureSuite API key (sk_…): ")
assert API_KEY.startswith("sk_"), "expected a key like sk_test_… or sk_live_… from the /developer page"
print("key loaded:", API_KEY.split("_")[0] + "_" + API_KEY.split("_")[1] + "_…")

## A tiny client

Everything below goes through this ~40-line helper. It adds the `Authorization`
header, honours `Retry-After` on `429` rate limits, follows cursor pagination,
and turns the API's error envelope `{"error": {"code", "message", "details"}}`
into a typed Python exception.


In [ ]:
import json, time, uuid
import requests

class SureSuiteError(RuntimeError):
    """An API error envelope: [status code] message (+ optional details)."""
    def __init__(self, status, code, message, details=None):
        super().__init__(f"[{status} {code}] {message}")
        self.status, self.code, self.message, self.details = status, code, message, details

class SureSuite:
    """Minimal client for the SureSuite public /v1 gateway."""

    def __init__(self, base_url, api_key):
        self.base = base_url.rstrip("/")
        self.session = requests.Session()
        self.session.headers["Authorization"] = f"Bearer {api_key}"

    def _call(self, method, path, retries=3, **kw):
        for attempt in range(retries + 1):
            r = self.session.request(method, self.base + path, timeout=60, **kw)
            if r.status_code == 429 and attempt < retries:
                wait = int(r.headers.get("Retry-After", "5"))
                print(f"  rate-limited — waiting {wait}s"); time.sleep(wait)
                continue
            break
        body = r.json() if r.content else {}
        if r.status_code >= 400:
            err = (body or {}).get("error", {})
            raise SureSuiteError(r.status_code, err.get("code", "unknown"),
                                 err.get("message", r.text[:200]), err.get("details"))
        return body

    def get(self, path, **params):
        return self._call("GET", path, params=params or None)

    def post(self, path, body=None, idempotency_key=None):
        headers = {"Idempotency-Key": idempotency_key} if idempotency_key else {}
        return self._call("POST", path, json=body or {}, headers=headers)

    def put(self, path, body):
        return self._call("PUT", path, json=body)

    def list_all(self, path, **params):
        """Follow cursor pagination (`?limit=&cursor=`) until exhausted."""
        rows, cursor = [], None
        while True:
            page = self.get(path, **params, **({"cursor": cursor} if cursor else {}))
            rows += page["data"]
            cursor = page.get("next_cursor")
            if not cursor:
                return rows

api = SureSuite(BASE_URL, API_KEY)

## 1 · Connect & list your projects

`GET /projects` returns only the projects your key's organization (and optional
per-project restriction) allows — a project outside your tenancy reads as `404`,
indistinguishable from one that doesn't exist.


In [ ]:
import pandas as pd

projects = api.list_all("/projects")
print(f"this key can see {len(projects)} project(s)")
pd.DataFrame(projects)[["id", "name", "supply_chain_model", "created_at"]]

In [ ]:
if not PROJECT_ID:
    PROJECT_ID = projects[0]["id"]   # default to the first visible project
project = api.get(f"/projects/{PROJECT_ID}")
print(f"working in: {project['name']}  ({PROJECT_ID})")

## 2 · Input data — dataset versions & freezing

A **dataset version** is an immutable, hash-stamped snapshot of the project's input
data (`graph_hash` is one corner of the provenance triangle every run records).
Freezing is **deduplicated server-side**: if nothing changed since the last
snapshot you get that same version back.


In [ ]:
dataset_versions = api.get(f"/projects/{PROJECT_ID}/dataset-versions")["data"]
pd.DataFrame(dataset_versions)

In [ ]:
# Freeze the current input data (requires write:data). Safe to re-run —
# an unchanged dataset returns the existing version instead of a new one.
frozen = api.post(f"/projects/{PROJECT_ID}/datasets:freeze", {"label": "notebook checkpoint"})
print("dataset version", frozen["id"], "graph_hash", (frozen.get("graph_hash") or "")[:16], "…")

## 3 · The engine policy catalog

`GET …/policy-catalog` returns the **registry export** — the single source of truth
for what the simulation engine supports: every policy, its catalog id (`P-S.x`
supplier, `P-P.x` plant, `P-T.x` transport, `P-C.x` customer, …), and a JSON
Schema of its parameters. The UI forms, the validation gate, and this API all
consume the same artifact, so what you see here is exactly what the engine runs.


In [ ]:
catalog = api.get(f"/projects/{PROJECT_ID}/policy-catalog")
print("engine", catalog["engine_version"], "—", len(catalog["policies"]), "policies")
pd.DataFrame([
    {"catalog_ref": p.get("catalog_ref"), "policy": p["id"], "stage": p.get("stage"),
     "params": len((p.get("params_schema") or {}).get("properties", {}))}
    for p in catalog["policies"]
])

In [ ]:
# Zoom into one policy's parameter schema — titles, types, bounds, defaults.
policy = next(p for p in catalog["policies"] if p["id"] == "unmet_demand_handling")
pd.DataFrame([
    {"param": name, "title": s.get("title"), "type": s.get("type"),
     "default": s.get("default"), "min": s.get("minimum"), "max": s.get("maximum"),
     "unit": s.get("unit")}
    for name, s in policy["params_schema"]["properties"].items()
])

## 4 · Read & edit the project's policy configuration

Policies live in two layers: **defaults** per family
(`sourcing · inventory · transport · fulfillment · production · recovery · demand`)
plus targeted **overrides** (per node / edge). `PUT …/policies` goes through the
exact same RPCs the app's /policies page uses — there is no separate API path
that could drift from the UI.


In [ ]:
policies = api.get(f"/projects/{PROJECT_ID}/policies")
defaults = policies["defaults"] or {}
print("families configured:", [k for k in defaults
                               if k in ("sourcing", "inventory", "transport",
                                        "fulfillment", "production", "recovery", "demand")])
print(f"{len(policies['overrides'])} per-node/edge override(s)")
print(json.dumps(defaults.get("inventory") or {}, indent=2)[:800])

In [ ]:
# Edit pattern: read the family, change what you need, write the family back.
# Parameter names and bounds come from the §3 catalog for your engine version.
edited_inventory = dict(defaults.get("inventory") or {})
# edited_inventory["<param>"] = <value>          # ← your experiment here

api.put(f"/projects/{PROJECT_ID}/policies", {"defaults": {"inventory": edited_inventory}})
print("policies saved")

# Targeted override example (uncomment and point at one of your nodes):
# api.put(f"/projects/{PROJECT_ID}/policies", {"overrides": [{
#     "scope": "node", "target_key": "PLANT_A", "family": "inventory",
#     "patch": {"<param>": <value>},
# }]})

## 5 · Snapshot an immutable policy version

Runs never execute against the *live* (editable) configuration — they run against
an immutable **policy version** identified by `policy_hash`. Snapshot before every
experiment so results stay reproducible and attributable.


In [ ]:
if not POLICY_VERSION_ID:
    snap = api.post(f"/projects/{PROJECT_ID}/policy-versions", {"label": "notebook baseline"})
    POLICY_VERSION_ID = snap["id"]
    print("snapshotted", POLICY_VERSION_ID, "policy_hash", (snap.get("policy_hash") or "")[:16], "…")

pd.DataFrame(api.get(f"/projects/{PROJECT_ID}/policy-versions")["data"])[
    ["id", "label", "policy_hash", "created_at"]].head(8)

## 6 · Scenarios

A **scenario** defines the experimental frame: horizon, warm-up, number of
replications, seed (with common random numbers for fair comparisons), and an
optional **disruption schedule**.


In [ ]:
scenarios = api.list_all(f"/projects/{PROJECT_ID}/scenarios")
pd.DataFrame(scenarios)[["id", "name", "horizon_days", "warmup_days", "replications", "seed"]]

In [ ]:
if not SCENARIO_ID:
    if scenarios:
        SCENARIO_ID = scenarios[0]["id"]
        print("using existing scenario:", scenarios[0]["name"])
    else:
        baseline = api.post(f"/projects/{PROJECT_ID}/scenarios", {
            "name": "Notebook baseline",
            "description": "Created by the API quickstart notebook",
            "horizon_days": 120, "warmup_days": 14,
            "replications": 10, "seed": 42, "crn": True,
            "primary_kpi": "fill_rate",
        })
        SCENARIO_ID = baseline["id"]
        print("created scenario:", SCENARIO_ID)

## 7 · Dispatch a run & poll to completion

Runs are **asynchronous**: `POST …/runs` answers `202` with a `run_id` immediately;
you poll `GET /runs/{id}` until `status ∈ {succeeded, failed, cancelled}`.

Three behaviours worth knowing:

- **Idempotency** — send an `Idempotency-Key`; retrying the same key within 24 h
  returns the *same* run instead of dispatching (and paying for) a duplicate.
- **`409 reuse_available`** — an identical completed run (same policy, data and
  scenario hashes) already exists. Read it, or force a recompute with
  `force_rerun=true`. Reuse is always *your* choice, never silent.
- **`422 validation_failed`** — the pre-run data gate found the input data
  incomplete; the findings tell you exactly what's missing.


In [ ]:
def run_and_wait(scenario_id, policy_version_id, idempotency_key=None,
                 poll_seconds=5, timeout_seconds=1800, **extra):
    """Dispatch a run and poll until it reaches a terminal state."""
    try:
        submitted = api.post(
            f"/projects/{PROJECT_ID}/runs",
            {"scenario_id": scenario_id, "policy_version_id": policy_version_id, **extra},
            idempotency_key=idempotency_key or f"nb-{uuid.uuid4()}",
        )
    except SureSuiteError as e:
        if e.code == "reuse_available":
            run_id = e.details["reuse_candidate"]["run_id"]
            print("identical completed run already exists — reusing", run_id)
            return api.get(f"/runs/{run_id}")
        if e.code == "validation_failed":
            print("blocked by the data-completeness gate:")
            for finding in (e.details or {}).get("findings") or []:
                print("  -", finding)
        raise

    run_id = submitted["run_id"]
    print("run", run_id, "dispatched — status", submitted["status"])
    started = time.time()
    while True:
        run = api.get(f"/runs/{run_id}")
        if run["status"] in ("succeeded", "failed", "cancelled"):
            print(f"\n→ {run['status']} after {time.time() - started:.0f}s")
            if run["status"] == "failed":
                print("error:", run.get("error_message"))
            return run
        done, target = run.get("rep_count_done") or 0, run.get("rep_count_target") or "?"
        print(f"  {run['status']} — replication {done}/{target}   ", end="\r")
        if time.time() - started > timeout_seconds:
            raise TimeoutError(f"run {run_id} still {run['status']} after {timeout_seconds}s")
        time.sleep(poll_seconds)

run = run_and_wait(SCENARIO_ID, POLICY_VERSION_ID, idempotency_key="nb-quickstart-baseline")

## 8 · Results — aggregate KPIs & per-replication analysis

The run row carries the **aggregate KPIs** (means across replications), the
**95 % CI half-widths**, and the full provenance triangle
(`policy_hash · graph_hash · scenario_hash`). Per-replication rows let you do your
own statistics.


In [ ]:
summary = pd.DataFrame({
    "mean": pd.Series(run["aggregate_kpis"] or {}),
    "ci_half_width_95": pd.Series(run["ci_half_widths"] or {}),
})
print("provenance:", {k: (run.get(k) or "")[:12] for k in ("policy_hash", "graph_hash", "scenario_hash")})
summary

In [ ]:
replications = api.list_all(f"/runs/{run['id']}/replications")
reps = pd.DataFrame([{"rep": r["rep_index"], "seed": r.get("seed_used"), **(r.get("kpis") or {})}
                     for r in replications])
reps.describe().T[["count", "mean", "std", "min", "max"]]

In [ ]:
import matplotlib.pyplot as plt

PALETTE = ["#2a78d6", "#008300", "#e87ba4", "#eda100"]  # fixed series order

primary_kpi = next((s.get("primary_kpi") for s in scenarios if s["id"] == SCENARIO_ID), None) or \
              (reps.columns[2] if len(reps.columns) > 2 else None)
if primary_kpi in reps:
    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.hist(reps[primary_kpi], bins=min(10, max(3, len(reps) // 2)),
            color=PALETTE[0], edgecolor="white")
    ax.set_title(f"{primary_kpi} across {len(reps)} replications")
    ax.set_xlabel(primary_kpi); ax.set_ylabel("replications")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.show()

In [ ]:
# Weekly time series per replication (opt-in — the payload is larger):
with_series = api.get(f"/runs/{run['id']}/replications", include="time_series", limit=3)["data"]
series = (with_series[0].get("time_series") or {}) if with_series else {}
print("series available:", list(series)[:12])

if primary_kpi in series and isinstance(series[primary_kpi], list):
    fig, ax = plt.subplots(figsize=(7, 3.2))
    for i, rep_row in enumerate(with_series):
        values = (rep_row.get("time_series") or {}).get(primary_kpi) or []
        ax.plot(range(len(values)), values, color=PALETTE[0],
                alpha=0.35 + 0.3 * (i == 0), linewidth=2)
    ax.set_title(f"{primary_kpi} — weekly trajectory (first {len(with_series)} replications)")
    ax.set_xlabel("week"); ax.set_ylabel(primary_kpi)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.show()

## 9 · Model-credibility status

Every run reports a credibility label — `validated` (an active model-validation
card matches this run's exact policy/data/scenario hashes), `stale` (a card
exists but the model changed since), or `unvalidated`. It's a **label, not a
gate**: the API tells you honestly how much to trust the numbers.


In [ ]:
validation = api.get(f"/runs/{run['id']}/validation")
print("credibility:", validation["status"])
card = validation.get("model_validation")
if card:
    print("verdict:", card.get("verdict"), "| validated at:", card.get("validated_at"),
          "| by:", card.get("author_email"))

## 10 · Experiment — compare policy A vs B

The core workflow: **edit → snapshot → run → compare**, with common random
numbers (CRN) making the comparison fair. Version A is the §5 baseline (already
run). For B, apply an edit in the cell below, snapshot, and run.

> If you make *no* edit, B's hashes equal A's and the dispatcher answers
> `409 reuse_available` — the notebook then reuses A's results, which neatly
> demonstrates the provenance-based run cache.


In [ ]:
# ── Variant B: apply an experimental edit (same pattern as §4) ──
variant_inventory = dict((api.get(f"/projects/{PROJECT_ID}/policies")["defaults"] or {}).get("inventory") or {})
# variant_inventory["<param>"] = <value>        # ← the change you want to test
api.put(f"/projects/{PROJECT_ID}/policies", {"defaults": {"inventory": variant_inventory}})

version_b = api.post(f"/projects/{PROJECT_ID}/policy-versions", {"label": "notebook variant B"})["id"]
run_a = run
run_b = run_and_wait(SCENARIO_ID, version_b, idempotency_key="nb-quickstart-variant-b")

In [ ]:
import numpy as np

kpi_names = sorted(set(run_a["aggregate_kpis"] or {}) & set(run_b["aggregate_kpis"] or {}))[:6]
if kpi_names:
    fig, axes = plt.subplots(1, len(kpi_names), figsize=(2.6 * len(kpi_names), 3.2))
    for ax, kpi in zip(np.atleast_1d(axes), kpi_names):   # one KPI per panel: scales differ
        values = [run_a["aggregate_kpis"][kpi], run_b["aggregate_kpis"][kpi]]
        errors = [(run_a.get("ci_half_widths") or {}).get(kpi) or 0,
                  (run_b.get("ci_half_widths") or {}).get(kpi) or 0]
        ax.bar(["A", "B"], values, yerr=errors, capsize=4,
               color=[PALETTE[0], PALETTE[1]], width=0.6)
        ax.set_title(kpi, fontsize=10)
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle("Policy A vs B — aggregate KPIs (error bars = 95% CI half-width)", y=1.02)
    plt.tight_layout(); plt.show()

# Restore the original configuration and re-snapshot if you edited anything:
# api.put(f"/projects/{PROJECT_ID}/policies", {"defaults": {"inventory": <original>}})

## 11 · Experiment — disruption resilience

Same policy, two scenarios: the baseline vs one with a **disruption schedule**
(e.g. a supplier knocked out for two weeks). CRN + the same seed make the delta
attributable to the disruption alone.

Set `DISRUPTED_NODE` to a node key from your network (see the app's network view,
or your item-master/location data).


In [ ]:
DISRUPTED_NODE = ""   # ← e.g. "SUPPLIER_A" — leave "" to skip this section

if DISRUPTED_NODE:
    disrupted_scenario = api.post(f"/projects/{PROJECT_ID}/scenarios", {
        "name": f"Notebook disruption — {DISRUPTED_NODE} outage",
        "description": "14-day full outage from day 30 (API quickstart §11)",
        "horizon_days": 120, "warmup_days": 14, "replications": 10, "seed": 42, "crn": True,
        "disruption_schedule": [{
            "target": DISRUPTED_NODE, "target_type": "node",
            "start_day": 30, "duration_days": 14, "magnitude_pct": 100,
        }],
    })
    run_disrupted = run_and_wait(disrupted_scenario["id"], POLICY_VERSION_ID,
                                 idempotency_key="nb-quickstart-disruption")

    kpis = sorted(set(run_a["aggregate_kpis"] or {}) & set(run_disrupted["aggregate_kpis"] or {}))[:6]
    fig, axes = plt.subplots(1, len(kpis), figsize=(2.6 * len(kpis), 3.2))
    for ax, kpi in zip(np.atleast_1d(axes), kpis):
        ax.bar(["baseline", "disrupted"],
               [run_a["aggregate_kpis"][kpi], run_disrupted["aggregate_kpis"][kpi]],
               yerr=[(run_a.get("ci_half_widths") or {}).get(kpi) or 0,
                     (run_disrupted.get("ci_half_widths") or {}).get(kpi) or 0],
               capsize=4, color=[PALETTE[0], PALETTE[3]], width=0.6)
        ax.set_title(kpi, fontsize=10)
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle("Baseline vs disrupted — aggregate KPIs", y=1.02)
    plt.tight_layout(); plt.show()
else:
    print("set DISRUPTED_NODE to run the resilience experiment")

## 12 · Operations — cancel, extend, errors

```python
api.post(f"/runs/{run['id']}:cancel")            # stop an in-flight run
api.post(f"/runs/{run['id']}:add-reps", {"n": 10})  # tighten CIs by adding replications
```

Error codes you'll actually meet (all share the `{"error": {…}}` envelope; every
response echoes an `X-Request-Id` you can quote to support):

| Code | Status | Meaning / what to do |
|---|---|---|
| `invalid_key` / `expired_key` / `revoked_key` | 401 | Fix or rotate the key on `/developer` |
| `missing_scope` | 403 | Recreate the key with the scope listed in the header table |
| `project_not_found` / `run_not_found` | 404 | Wrong id — or outside your key's tenancy (deliberately indistinguishable) |
| `validation_failed` | 422 | Input data incomplete; `details.findings` says what's missing |
| `reuse_available` | 409 | Identical completed run exists; reuse it or `force_rerun=True` |
| `rate_limited` / `daily_quota_exceeded` / `concurrent_runs_exceeded` | 429 | Back off (`Retry-After`); test keys allow 1 concurrent run |

**Where to go next**

- Full endpoint reference: `docs/api/README.md` in the repository.
- Key management, scopes, usage: the app's **`/developer`** page.
- Treat the API key like a password — Colab Secrets or an environment variable,
  never a notebook cell, never source control.
